# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [1]:
# Uncomment in a fresh Kaggle notebook environment.
# %pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml

In [1]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

/home/william-zheng/Documents/Programming/Python/DL_course/Midterm/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.9.1+cu128
CUDA available: True


In [2]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    # "model_name": "unsloth/Qwen3.5-2B-Instruct-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "model_name": "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    # https://huggingface.co/unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 12000,
    "eval_size": 0.02,
    # "output_dir": "/kaggle/working/qwen2b_svg_lora",
    "output_dir": "./sloth_output/working/qwen2b_svg_lora",
}

CONFIG

{'model_name': 'unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit',
 'max_seq_length': 2048,
 'lora_r': 16,
 'lora_alpha': 16,
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 2,
 'gradient_accumulation_steps': 8,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 12000,
 'eval_size': 0.02,
 'output_dir': './sloth_output/working/qwen2b_svg_lora'}

In [3]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
DATASET_CATALOG = {
    "OmniSVG/MMSVG-Icon": {
        "split": "train",
        "prompt_fields": ["description", "keywords", "detail", "prompt", "text"],
        "svg_fields": ["svg", "picosvg", "completion", "target"],
    },
    "xingxm/SVGX-Core-250k": {
        "split": "train",
        "prompt_fields": ["qwen_caption", "blip_caption", "name", "img_analysis", "prompt"],
        "svg_fields": ["svg", "completion", "target"],
    },
    "xingxm/SVGX-SFT-1M": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input", "query"],
        "svg_fields": ["completion", "output", "svg", "response"],
    },
    "thesantatitan/deepseek-svg-dataset": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input"],
        "svg_fields": ["completion", "output", "svg"],
    },
}

# For a first run, keep to 1-2 sources.
ACTIVE_SOURCES = [
    # "xingxm/SVGX-SFT-1M",
    # "OmniSVG/MMSVG-Icon",
]

In [4]:
def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""


def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)
    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    return {"prompt": prompt, "svg": svg}


def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")
    ds = load_dataset(dataset_id, split=cfg["split"])
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"{dataset_id}: {len(ds)} usable rows")
    return ds

In [5]:
# # MY DATA LOADING
# from unsloth import FastLanguageModel

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=CONFIG["model_name"],
#     max_seq_length=CONFIG["max_seq_length"],
#     dtype=None,
#     load_in_4bit=True,
# )

# def format_svg_sample(prompt: str, svg_code: str) -> str:
#     """Format a prompt-SVG pair into the chat template expected by the model."""
#     messages = [
#         {"role": "system", "content": "You are an expert SVG code generator. Generate clean, valid SVG code based on the user's description."},
#         {"role": "user", "content": prompt},
#         {"role": "assistant", "content": svg_code},
#     ]
#     return tokenizer.apply_chat_template(messages, tokenize=False)


# import pandas as pd
# from tqdm import tqdm
from pathlib import Path

DATA = Path('./dataset')
# DATA = Path('/content/dataset')

train_df = pd.read_csv(DATA / 'train.csv')

# new_prompt = []
# for r in tqdm(range(train_df.shape[0])):
    # new_prompt.append(format_svg_sample(train_df.loc[r, 'prompt'], train_df.loc[r, 'svg']))
# train_df['text'] = new_prompt

train_df = train_df.iloc[0:1000]

from datasets import Dataset
# from datasets import load_dataset


# train_dataset = load_dataset("csv", data_files=DATA / "train.csv")
# train_dataset = Dataset.from_pandas(train_df[['text']])
train_dataset = Dataset.from_pandas(train_df)
train_dataset

Dataset({
    features: ['id', 'prompt', 'svg'],
    num_rows: 1000
})

In [6]:
# FOR LOGGIN IN ON HF ON COLAB

# from google.colab import userdata
# from huggingface_hub import login

# # Get the token from Colab secrets
# HF_TOKEN = userdata.get('HF_TOKEN')

# # Log in to Hugging Face
# if HF_TOKEN:
#     login(token=HF_TOKEN)
#     print("Successfully logged in to Hugging Face!")
# else:
#     print("Token is not set. Please save the token first.")


In [7]:
# COMMENTED OUT TO US KAGGLE DATASET
# datasets_ok = []
# for source in ACTIVE_SOURCES:
#     try:
#         ds = load_source_dataset(
#             source,
#             DATASET_CATALOG[source],
#             CONFIG["max_train_samples_per_source"],
#         )
#         datasets_ok.append(ds)
#     except Exception as e:
#         print(f"Skipping {source}: {type(e).__name__}: {e}")

# if not datasets_ok:
#     raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")
datasets_ok = [train_dataset]

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Train rows: 980
Eval rows: 20


{'id': '2754e2967ce866bf3d9c002a9bef5aab',
 'prompt': 'A simple organizational chart with a central node connected to three other nodes.',
 'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#666666" fill-opacity="1.0"  filling="0" d="M29.188995361328125 102.0780029296875 L95.84500122070312 102.0780029296875 L95.84500122070312 123.0999984741211 L104.18900299072266 123.0999984741211 L104.18900299072266 102.0780029296875 L170.84500122070312 102.0780029296875 L170.84500122070312 127.30000305175781 L179.17799377441406 127.30000305175781 L179.17799377441406 93.6780014038086 L104.18900299072266 93.6780014038086 L104.18900299072266 76.8550033569336 L95.84500122070312 76.8550033569336 L95.84500122070312 93.6780014038086 L20.854995727539062 93.6780014038086 L20.854995727539062 127.30000305175781 L29.188995361328125 127.30000305175781 L29.188995361328125 102.0780029296875 Z M45.845001220703125 135.71099853515625 L4.189002990

In [8]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

Map: 100%|██████████| 20/20 [00:00<00:00, 10854.82 examples/s]

<|im_start|>system
You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element.<|im_end|>
<|im_start|>user
A simple organizational chart with a central node connected to three other nodes.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#666666" fi


In [9]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.10: Fast Qwen3_Vl patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.633 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 625/625 [00:00<00:00, 3657.32it/s]


Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [11]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)

train_result = trainer.train()
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Unsloth: Tokenizing ["text"] (num_proc=16): 100%|██████████| 20/20 [00:11<00:00,  1.75 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 980 | Num Epochs = 1 | Total steps = 62
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 17,432,576 of 2,144,964,608 (0.81% trained)


Step,Training Loss,Validation Loss


TrainOutput(global_step=62, training_loss=0.6019618270858642, metrics={'train_runtime': 486.5128, 'train_samples_per_second': 2.014, 'train_steps_per_second': 0.127, 'total_flos': 1.081754914332672e+16, 'train_loss': 0.6019618270858642, 'epoch': 1.0})

In [15]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

NameError: name 'trainer' is not defined

In [11]:
from unsloth import FastLanguageModel

# LOading Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["output_dir"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.10: Fast Qwen3_Vl patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.633 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 625/625 [00:00<00:00, 3330.19it/s]


In [10]:
FastLanguageModel.for_inference(model)

SVG_REGEX = re.compile(r"<svg[\\s\\S]*?</svg>", flags=re.IGNORECASE)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=512):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(text=chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded)
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)
    return svg


test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt)
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))

<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="black"/></svg>
Valid SVG: True


In [12]:
from tqdm import tqdm

def generate_svg_batch(all_prompts:list[str], max_new_tokens=512, batchsize=8):
    all_svgs = []

    for p_i in tqdm(range(0, len(all_prompts), batchsize), desc='Svg To Prompts'):
        chat_texts = [
            "<|im_start|>system\n"
            f"{SYSTEM_PROMPT}<|im_end|>\n"
            "<|im_start|>user\n"
            f"{prompt}<|im_end|>\n"
            "<|im_start|>assistant\n" for prompt in all_prompts[p_i : p_i + batchsize]
        ]
        

        inputs = tokenizer(
            text=chat_texts, 
            padding=True, 
            return_tensors="pt",
            padding_side='left', # TODO: UNDERSTAND PADDING EFFECT
        ).to(model.device)
        
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.05,
            )
        decoded_prompts = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        for i in range(len(decoded_prompts)):
            svg = extract_svg(decoded_prompts[i])
            if not is_valid_svg(svg):
                svg = fallback_svg( 'NULL' )
            all_svgs += svg

    return all_svgs

In [13]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
# TEST_PROMPTS_PATH = "/kaggle/input/svg-test-public-prompts/test_prompts.csv"
# SUBMISSION_PATH = "/kaggle/working/submission.csv"
TEST_PROMPTS_PATH = "./dataset/test.csv"
SUBMISSION_PATH = "./dataset/sample_submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

rows = []
invalid_count = 0
t0 = time.time()

for _, row in test_df.iterrows():
    svg = generate_svg(row["prompt"])
    if not is_valid_svg(svg):
        invalid_count += 1
        svg = fallback_svg(row["prompt"])
    rows.append({"id": row["id"], "svg": svg})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

KeyboardInterrupt: 

In [ ]:
import gc
gc.collect()
# Clear PyTorch's GPU memory cache
torch.cuda.empty_cache()

In [ ]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
# TEST_PROMPTS_PATH = "/kaggle/input/svg-test-public-prompts/test_prompts.csv"
# SUBMISSION_PATH = "/kaggle/working/submission.csv"
TEST_PROMPTS_PATH = "./dataset/test.csv"
SUBMISSION_PATH = "./dataset/sample_submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

rows = []
invalid_count = 0
t0 = time.time()

all_svgs = generate_svg_batch(
    test_df['prompt'].tolist(),
    batchsize=90
)
for i in range(len(all_svgs)):
    rows.append({"id": test_df.loc[i, "id"], "svg": all_svgs[i]})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

Svg To Prompts:   0%|          | 0/12 [00:01<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 120.00 MiB. GPU 0 has a total capacity of 11.63 GiB of which 73.56 MiB is free. Including non-PyTorch memory, this process has 11.54 GiB memory in use. Of the allocated memory 10.66 GiB is allocated by PyTorch, and 741.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.